In [1]:
!pip install transformers peft rouge-score python-Levenshtein tqdm scikit-learn

In [2]:
import json
import torch
import numpy as np

from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

from rouge_score import rouge_scorer
import Levenshtein

In [3]:
BASE_MODEL = "microsoft/phi-2"

MODEL_P_PATH = "/Users/amarsinha/Research_swifties/models/Model_P"

DATA_PARA = "/Users/amarsinha/Research_swifties/data/paraphrases/paraphrased_200.json"

PREFIX_RATIOS = [0.5]

In [4]:
def load_model(adapter_path):

    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

    tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        torch_dtype=torch.float16
    )

    model = PeftModel.from_pretrained(model, adapter_path)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)

    model.config.pad_token_id = tokenizer.eos_token_id
    model.base_model.config.pad_token_id = tokenizer.eos_token_id
    model.generation_config.pad_token_id = tokenizer.eos_token_id

    model.eval()

    return model, tokenizer

In [5]:
def load_dataset():

    with open(DATA_PARA) as f:
        data = json.load(f)

    return data

In [6]:
model_v, tokenizer = load_model(MODEL_P_PATH)

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/453 [00:00<?, ?it/s]

In [7]:
dataset = load_dataset()

dataset = dataset

In [8]:
def generate_completion(model, tokenizer, prefix):

    device = next(model.parameters()).device

    inputs = tokenizer(prefix, return_tensors="pt").to(device)

    with torch.no_grad():

        output = model.generate(
            **inputs,
            max_new_tokens=8,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
            do_sample=False,
            num_beams=1,
            use_cache=True
        )

    text = tokenizer.decode(output[0], skip_special_tokens=True)

    completion = text[len(prefix):]

    return completion

In [9]:
scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)

def rouge_l(pred, truth):

    score = scorer.score(truth, pred)

    return score['rougeL'].fmeasure


def normalized_edit(pred, truth):

    dist = Levenshtein.distance(pred, truth)

    norm = 1 - dist / max(len(truth), 1)

    return norm

In [10]:
def prefix_score_fast(model, tokenizer, text):

    scores = []

    for r in PREFIX_RATIOS:

        split = int(len(text) * r)

        prefix = text[:split]
        suffix = text[split:]

        full_text = prefix + suffix

        inputs = tokenizer(full_text, return_tensors="pt").to(next(model.parameters()).device)

        with torch.no_grad():
            outputs = model(**inputs, labels=inputs["input_ids"])

        loss = outputs.loss.item()

        score = 1 / (1 + loss)

        scores.append(score)

    return np.mean(scores)

In [11]:
dataset = load_dataset()

scores = []

for i, ex in enumerate(dataset):

    text = ex["question"]

    score = prefix_score_fast(model_v, tokenizer, text)

    scores.append(score)

    if (i+1) % 10 == 0:
        print(f"{i+1}/{len(dataset)} completed")

import pandas as pd

df = pd.DataFrame({
    "score": scores
})

df.to_csv("prefix_scores_model_p_paraphrase.csv", index=False)

print("Saved.")

10/200 completed
20/200 completed
30/200 completed
40/200 completed
50/200 completed
60/200 completed
70/200 completed
80/200 completed
90/200 completed
100/200 completed
110/200 completed
120/200 completed
130/200 completed
140/200 completed
150/200 completed
160/200 completed
170/200 completed
180/200 completed
190/200 completed
200/200 completed
Saved.
